<a href="https://colab.research.google.com/github/Ronaa-MX/simulacion-esfm/blob/simulacion-I/Ejemplo2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Máquinas averíadas

In [1]:
# Librerías a usar
import random as rd
import numpy as np
from scipy import stats as st
import matplotlib.pyplot as plot
from multiprocessing import Pool, cpu_count
import time

In [2]:
# ------------------------------------------------------------
# 1-3. Variables, probabilidades y CDFs
# ------------------------------------------------------------
tiempos_ind = np.array([20, 30, 40, 50, 60, 70, 80])
probs_ind   = np.array([0.05, 0.15, 0.15, 0.20, 0.20, 0.15, 0.10])
tiempos_grupo = np.array([30, 40, 50, 60, 70, 80, 90])
probs_grupo   = np.array([0.05, 0.15, 0.15, 0.20, 0.20, 0.15, 0.10])

cdf_ind = np.cumsum(probs_ind)
cdf_grupo = np.cumsum(probs_grupo)

In [3]:
# ------------------------------------------------------------
# 4. Generación vectorizada de tiempos
# ------------------------------------------------------------
def generar_tiempos(n, tiempos, cdf):
    u = np.random.uniform(0, 1, n)
    idx = np.searchsorted(cdf, u)
    return tiempos[idx]

In [4]:
# ------------------------------------------------------------
# 5. Experimento: simular N ciclos y devolver coste medio por hora
# ------------------------------------------------------------
def simular_politica(tiempos, cdf, t_repara, n_herramientas, n_ciclos):
    t_ops = generar_tiempos(n_ciclos, tiempos, cdf)
    tiempo_total = np.sum(t_ops) + n_ciclos * t_repara
    coste_total = 100 * n_ciclos * t_repara + n_herramientas * 10 * n_ciclos
    return coste_total / tiempo_total

In [5]:
# Función auxiliar para paralelización
def experimento(args):
    politica, n_ciclos = args
    if politica == 'ind':
        return simular_politica(tiempos_ind, cdf_ind, 1, 1, n_ciclos)
    else:
        return simular_politica(tiempos_grupo, cdf_grupo, 2, 5, n_ciclos)

In [6]:
# ------------------------------------------------------------
# Parámetros
# ------------------------------------------------------------
N_CICLOS_POR_SIM = 10000   # ciclos por experimento
N_SIMULACIONES = 1000          # número de experimentos (M)

# ------------------------------------------------------------
# 7. Simulaciones en paralelo
# ------------------------------------------------------------
def run_parallel(politica, n_sim, n_ciclos):
    args = [(politica, n_ciclos)] * n_sim
    with Pool(processes=cpu_count()) as pool:
        return np.array(pool.map(experimento, args))

In [ ]:
if __name__ == "__main__":
    start = time.time()
    print("Simulando política individual...")
    res_ind = run_parallel('ind', N_SIMULACIONES, N_CICLOS_POR_SIM)
    print("Simulando política grupal...")
    res_grupo = run_parallel('grupo', N_SIMULACIONES, N_CICLOS_POR_SIM)
    print(f"Tiempo total: {time.time()-start:.2f} s")
    
    # 9. Estadísticos
    media_ind, std_ind, var_ind = np.mean(res_ind), np.std(res_ind), np.var(res_ind)
    media_grupo, std_grupo, var_grupo = np.mean(res_grupo), np.std(res_grupo), np.var(res_grupo)
    
    # 10. Intervalos de confianza
    z = 1.96
    ic_ind = (media_ind - z*std_ind/np.sqrt(N_SIMULACIONES), media_ind + z*std_ind/np.sqrt(N_SIMULACIONES))
    ic_grupo = (media_grupo - z*std_grupo/np.sqrt(N_SIMULACIONES), media_grupo + z*std_grupo/np.sqrt(N_SIMULACIONES))
    
    print("\n===== RESULTADOS =====")
    print(f"Individual: media = {media_ind:.2f} €/h, std = {std_ind:.2f}, var = {var_ind:.2f}")
    print(f"  IC 95% media: ({ic_ind[0]:.2f}, {ic_ind[1]:.6f})")
    print(f"Grupal: media = {media_grupo:.2f} €/h, std = {std_grupo:.2f}, var = {var_grupo:.2f}")
    print(f"  IC 95% media: ({ic_grupo[0]:.2f}, {ic_grupo[1]:.2f})")
    
    # 8. Histogramas
    plt.figure(figsize=(12,5))
    plt.subplot(1,2,1)
    plt.hist(res_ind, bins=30, alpha=0.7, color='blue', edgecolor='black')
    plt.axvline(media_ind, color='red', linestyle='--', label=f'Media = {media_ind:.4f}')
    plt.xlabel('Coste medio por hora (€/h)'); plt.ylabel('Frecuencia')
    plt.title('Política individual'); plt.legend()
    
    plt.subplot(1,2,2)
    plt.hist(res_grupo, bins=30, alpha=0.7, color='green', edgecolor='black')
    plt.axvline(media_grupo, color='red', linestyle='--', label=f'Media = {media_grupo:.4f}')
    plt.xlabel('Coste medio por hora (€/h)'); plt.ylabel('Frecuencia')
    plt.title('Política grupal'); plt.legend()
    plt.tight_layout()
    plt.show()
    
    # 6. Gráfico de convergencia (opcional)
    def convergencia(tiempos, cdf, t_repara, n_herra, max_ciclos, paso=10000):
        costes = []; t_acum = 0; c_acum = 0
        for ciclo in range(1, max_ciclos+1):
            t_op = generar_tiempos(1, tiempos, cdf)[0]
            t_acum += t_op + t_repara
            c_acum += 100*t_repara + n_herra*10
            if ciclo % paso == 0:
                costes.append(c_acum / t_acum)
        return costes
    
    plt.figure(figsize=(10,5))
    conv_ind = convergencia(tiempos_ind, cdf_ind, 1, 1, 500000, 10000)
    conv_grupo = convergencia(tiempos_grupo, cdf_grupo, 2, 5, 500000, 10000)
    plt.plot(range(10000, 500001, 10000), conv_ind, label='Individual')
    plt.plot(range(10000, 500001, 10000), conv_grupo, label='Grupal')
    plt.axhline(y=media_ind, color='blue', linestyle='--', label=f'Media individual {media_ind:.4f}')
    plt.axhline(y=media_grupo, color='orange', linestyle='--', label=f'Media grupal {media_grupo:.4f}')
    plt.xlabel('Número de ciclos'); plt.ylabel('Coste medio (€/h)')
    plt.title('Convergencia de la simulación'); plt.legend(); plt.grid(alpha=0.3)
    plt.show()